In [1]:
import pandas as pd
import tqdm, os
import pickle
from sklearn.preprocessing import StandardScaler
import numpy as np
import warnings
warnings.simplefilter("ignore")

import sys
sys.path.insert(0, "../..")

from utils.data_processing import dataset_df_to_token_df
from utils.acoustics_processing import get_mfccs
from utils.articulation_processing import get_aeema_articulatory_features

In [2]:
df = pd.read_csv("../data_processing/dataset_construction/datasets/processed_aeema.csv")
df["subject"] = df["art_f"].apply(lambda x: os.path.basename(x).split("_")[2])
token_df = dataset_df_to_token_df(df, unit_type="syllable")

In [3]:
token_df

,start,end,syllable,art_f,ac_f,subject
0,4.5300,4.6500,n_i,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/wav/u...,f1
1,5.0100,5.2200,k_ɪ_ŋ,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/wav/u...,f1
2,4.3800,4.5300,m_ʌ,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/wav/u...,f1
3,11.4800,11.7100,m_ɝ_z,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/wav/u...,f1
4,8.0400,8.2400,n_ɝ,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/wav/u...,f1
...,...,...,...,...,...,...
13395,13.2589,13.4289,p_ɝ,/Volumes/Transcend/USC-TIMIT/EMA/Data/M3/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/M3/wav/u...,m3
13396,3.3189,3.3789,ɑ,/Volumes/Transcend/USC-TIMIT/EMA/Data/M3/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/M3/wav/u...,m3
13397,6.7389,6.8189,f_i,/Volumes/Transcend/USC-TIMIT/EMA/Data/M3/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/M3/wav/u...,m3
13398,14.2789,14.4589,ɹ_aɪ_d,/Volumes/Transcend/USC-TIMIT/EMA/Data/M3/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/M3/wav/u...,m3


In [4]:
syllable_mean_duration = (token_df["end"] - token_df["start"]).mean()
phone_df = pd.read_csv("../../phone_based/data_processing/dataset_construction/datasets/processed_aeema.csv")
phone_df["subject"] = phone_df["art_f"].apply(lambda x: os.path.basename(x).split("_")[2])
phone_token_df = dataset_df_to_token_df(phone_df, unit_type="phone")
phone_mean_duration = (phone_token_df["end"] - phone_token_df["start"]).mean()

In [7]:
target_num_frames = round(10 * syllable_mean_duration / phone_mean_duration)

In [11]:
for subject in token_df["subject"].unique():
    os.makedirs(f"aeema_scalers", exist_ok=True)
    if os.path.exists(f"aeema_scalers/{subject}.pkl"):
        continue

    subject_token_df = token_df[token_df["subject"]==subject]

    data = {}

    for start, end, art_f in tqdm.tqdm(zip(subject_token_df["start"], subject_token_df["end"], subject_token_df["art_f"]), total=len(subject_token_df)):
        features = get_aeema_articulatory_features(art_f, start, end, target_num_frames=target_num_frames)

        if features is None:
            continue
        for idx in range(features.shape[1]):
            if idx not in data:
                data[idx] = []
            data[idx].append(features[:, idx])

    scalers = {}
    for idx in tqdm.tqdm(data, total=len(data)):
        data[idx] = np.concatenate(data[idx]).reshape(-1, 1)
        scaler = StandardScaler()
        scaler.fit(data[idx])
        scalers[idx] = scaler

    with open(f"aeema_scalers/{subject}.pkl", "wb") as f:
        pickle.dump(scalers, f)

100%|██████████| 18/18 [00:00<00:00, 854.01it/s]


In [12]:
scalers = {}
for subject in token_df["subject"].unique():
    with open(f"aeema_scalers/{subject}.pkl", "rb") as f:
        scalers[subject] = pickle.load(f)

In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from utils.models import SequenceAutoencoder

In [14]:
def temporal_diff(x):
    """
    x: (B, T, ...)
    returns: (B, T-1, ...)
    """
    return x[:, 1:] - x[:, :-1]

def combined_loss(x_hat, x, alpha=0.5):
    """
    alpha: weight on static loss (delta weight = 1-alpha)
    """
    # static loss
    loss_static = F.mse_loss(x_hat, x)

    # delta loss
    dx_hat = temporal_diff(x_hat)
    dx = temporal_diff(x)

    loss_delta = F.mse_loss(dx_hat, dx)

    return alpha * loss_static + (1 - alpha) * loss_delta, loss_static, loss_delta

def latent_stats(z):
    # z: (B, latent_dim)
    return {
        "latent_var_mean": z.var(dim=0).mean().item(),
        "latent_var_min":  z.var(dim=0).min().item(),
        "latent_var_max":  z.var(dim=0).max().item(),
    }

def batch_diversity(x):
    # x: (B, ...)
    x_flat = x.reshape(x.size(0), -1)
    return x_flat.std(dim=0).mean().item()

In [15]:
device = "mps" if torch.backends.mps.is_available() else "cpu"

BATCH_SIZE = 4

In [ ]:
log = {
    "batch_idx": [],
    "loss": [],
}
art_model = SequenceAutoencoder(
    feature_shape=(6, 3),
    latent_dim=1024,
).to(device)

art_criterion = nn.MSELoss()
art_optimizer = torch.optim.Adam(art_model.parameters(), lr=0.001)

shuffled_token_df = token_df.sample(frac=1, random_state=42).reset_index(drop=True)

pbar = tqdm.tqdm(shuffled_token_df.iterrows(), total=len(shuffled_token_df))

art_model.train()

batch = []
batch_idx = 0
for idx, (_, data) in enumerate(pbar):

    start, end, syllable, art_f, ac_f, subject = data["start"], data["end"], data["syllable"], data["art_f"], data["ac_f"], data["subject"]
    
    articulatory_features = get_aeema_articulatory_features(art_f, start, end, scalers=scalers[subject], target_num_frames=target_num_frames)

    if articulatory_features is None:
        continue

    articulatory_features = articulatory_features.reshape(articulatory_features.shape[0], 6, 3)

    batch.append(
        torch.tensor(articulatory_features, dtype=torch.float32)
    )

    # --------------------------------------------------
    # When batch is ready
    # --------------------------------------------------
    if len(batch) < BATCH_SIZE:
        continue
    batch_idx += 1
    art_batch = torch.stack(batch).to(device)  # (B,T,P,C)
    batch.clear()

    # ===== Articulatory =====
    art_out, art_x = art_model(art_batch)
    art_loss, art_static, art_delta = combined_loss(
        art_out, art_batch
    )

    art_optimizer.zero_grad()
    art_loss.backward()
    art_optimizer.step()

    log["batch_idx"].append(batch_idx)
    log["loss"].append(art_loss.item())
    pbar.set_description(
        f"{batch_idx}: ART {art_loss.item():.4f}"
    )

    # --------------------------------------------------
    # Diagnostics every 1000 updates
    # --------------------------------------------------
    if (batch_idx) % 300 == 0:
        with torch.no_grad():
            art_stats = latent_stats(art_x)

        print(
            f"[ART] latent var mean={art_stats['latent_var_mean']:.4e}, "
            f"min={art_stats['latent_var_min']:.4e}"
        )

        print(
            f"[ART] input std={batch_diversity(art_batch):.4e}, "
            f"recon std={batch_diversity(art_out):.4e}"
        )

        torch.save(art_model.state_dict(), "ckpts/aeema/art_model_ckpt.pth")
        pd.DataFrame(log).to_csv("ckpts/aeema/art_model_training_log.csv", index=False)

torch.save(art_model.state_dict(), "ckpts/aeema/art_model_ckpt.pth")
pd.DataFrame(log).to_csv("ckpts/aeema/art_model_training_log.csv", index=False)

In [ ]:
BATCH_SIZE = 4

In [ ]:
log = {
    "batch_idx": [],
    "loss": [],
}
ac_model = SequenceAutoencoder(
    feature_shape=30,
    latent_dim=1024,
).to(device)

ac_criterion = nn.MSELoss()
ac_optimizer = torch.optim.Adam(ac_model.parameters(), lr=0.001)

shuffled_token_df = token_df.sample(frac=1, random_state=42).reset_index(drop=True)

pbar = tqdm.tqdm(shuffled_token_df.iterrows(), total=len(shuffled_token_df))

ac_model.train()

batch  = []
batch_idx = 0
for idx, (_, data) in enumerate(pbar):

    start, end, syllable, art_f, ac_f, subject = data["start"], data["end"], data["syllable"], data["art_f"], data["ac_f"], data["subject"]
    
    mfccs = get_mfccs(ac_f, start, end).T

    if mfccs is None:
        continue

    batch.append(
        torch.tensor(mfccs, dtype=torch.float32)
    )

    # --------------------------------------------------
    # When batch is ready
    # --------------------------------------------------
    if len(batch) < BATCH_SIZE:
        continue
    batch_idx += 1

    ac_batch = torch.stack(batch).to(device)  # (B,T,F)
    batch.clear()

    # ===== Acoustic =====
    ac_out, ac_z = ac_model(ac_batch)
    ac_loss, ac_static, ac_delta = combined_loss(
        ac_out, ac_batch
    )

    ac_optimizer.zero_grad()
    ac_loss.backward()
    ac_optimizer.step()

    log["batch_idx"].append(batch_idx)
    log["loss"].append(ac_loss.item())
    pbar.set_description(
        f"{batch_idx}: AC {ac_loss.item():.4f}"
    )

    # --------------------------------------------------
    # Diagnostics every 1000 updates
    # --------------------------------------------------
    if batch_idx % 1000 == 999:
        with torch.no_grad():
            ac_stats  = latent_stats(ac_z)

        print(
            f"[AC ] latent var mean={ac_stats['latent_var_mean']:.4e}, "
            f"min={ac_stats['latent_var_min']:.4e}"
        )
        print(
            f"[AC ] input std={batch_diversity(ac_batch):.4e}, "
            f"recon std={batch_diversity(ac_out):.4e}"
        )

        torch.save(ac_model.state_dict(), "ckpts/aeema/ac_model_ckpt.pth")
        pd.DataFrame(log).to_csv("ckpts/aeema/ac_model_training_log.csv", index=False)

torch.save(ac_model.state_dict(), "ckpts/aeema/ac_model_ckpt.pth")
pd.DataFrame(log).to_csv("ckpts/aeema/ac_model_training_log.csv", index=False)